# Frontier Model Benchmark — RIAWELC Weld Defects

Benchmark GPT-4.1 and GPT-5 on 4-class weld defect classification from X-ray radiographs.

**Models:** GPT-4.1 (t=0.7), GPT-5 (t=1)

**Test set:** 240 images (60 per class, sampled from 2,443 test images)

In [1]:
import os, json, re, time, base64, random
import numpy as np
from PIL import Image
from collections import defaultdict, Counter
import matplotlib.pyplot as plt

# !pip install openai -q

## Config

In [7]:
from openai import AzureOpenAI
from config import CLASSES, REF_IMAGE_PATH, TEST_IMAGES_DIR
from config import make_prompt_zs, make_prompt_fs_ref, make_prompt_fs_query

GPT41_ENDPOINT = 'https://ether-openai.openai.azure.com/'
GPT41_DEPLOYMENT = 'gpt-4.1'
GPT41_API_VERSION = '2024-12-01-preview'
GPT41_API_KEY = '871a30b46ebf4297a8938ff5fca23646'

GPT5_ENDPOINT = 'https://ether-project-resource.openai.azure.com/'
GPT5_DEPLOYMENT = 'gpt-5'
GPT5_API_VERSION = '2024-12-01-preview'
GPT5_API_KEY = 'EeOsnTON1q9cOr8y90JuN5PpDNPW3ec3zKI0eCap8tdAfhR0e8TqJQQJ99BJACHYHv6XJ3w3AAAAACOGG63R'

client_41 = AzureOpenAI(azure_endpoint=GPT41_ENDPOINT, api_key=GPT41_API_KEY, api_version=GPT41_API_VERSION)
client_5 = AzureOpenAI(azure_endpoint=GPT5_ENDPOINT, api_key=GPT5_API_KEY, api_version=GPT5_API_VERSION)

print(f'GPT-4.1: key set: {bool(GPT41_API_KEY)}')
print(f'GPT-5:   key set: {bool(GPT5_API_KEY)}')

GPT-4.1: key set: True
GPT-5:   key set: True


## Load Data

In [8]:
random.seed(42)
SAMPLE_PER_CLASS = 60

manifest = []
for cls in CLASSES:
    cls_dir = os.path.join(TEST_IMAGES_DIR, cls)
    if not os.path.isdir(cls_dir):
        print(f'WARNING: {cls_dir} not found'); continue
    images = sorted([f for f in os.listdir(cls_dir) if f.endswith('.png')])
    if len(images) > SAMPLE_PER_CLASS:
        images = random.sample(images, SAMPLE_PER_CLASS)
    for img in images:
        manifest.append({'image': os.path.join(cls_dir, img), 'class': cls})

random.shuffle(manifest)
print(f'Test images: {len(manifest)}')
cls_counts = Counter(e['class'] for e in manifest)
for cls in CLASSES:
    print(f'  {cls}: {cls_counts.get(cls, 0)}')

# Load reference image
ref_b64 = None
if os.path.exists(REF_IMAGE_PATH):
    with open(REF_IMAGE_PATH, 'rb') as f:
        ref_b64 = base64.b64encode(f.read()).decode('utf-8')
    print(f'\nReference image loaded: {REF_IMAGE_PATH}')
else:
    print(f'WARNING: {REF_IMAGE_PATH} not found')

Test images: 240
  lack_of_penetration: 60
  porosity: 60
  cracks: 60
  no_defect: 60

Reference image loaded: riawelc_reference_grid.png


## Helpers

In [9]:
def encode_image(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def call_api(client, deployment, image_b64, prompt, ref_b64=None, ref_prompt=None, temperature=None):
    content = []
    if ref_b64 and ref_prompt:
        content.append({'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{ref_b64}'}})
        content.append({'type': 'text', 'text': ref_prompt})
    content.append({'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{image_b64}'}})
    content.append({'type': 'text', 'text': prompt})
    kwargs = dict(model=deployment, max_completion_tokens=8192,
                  messages=[{'role': 'user', 'content': content}])
    if temperature is not None:
        kwargs['temperature'] = temperature
    resp = client.chat.completions.create(**kwargs)
    return resp.choices[0].message.content

def parse_response(raw):
    if not raw: return None
    raw = raw.replace('<','').replace('>','')
    raw = re.sub(r'```json\s*', '', raw)
    raw = re.sub(r'```\s*', '', raw).strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict): return obj
    except: pass
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    dm = re.search(r'"defect_class"\s*:\s*"([\w]+)"', raw)
    if dm: return {'defect_class': dm.group(1)}
    raw_lower = raw.lower()
    for cls in sorted(CLASSES, key=len, reverse=True):
        if cls in raw_lower: return {'defect_class': cls}
    return None

print('Helpers ready.')

Helpers ready.


## Benchmark Function

In [10]:
def run_frontier_benchmark(manifest, client, deployment, mode, temperature=None, ref_b64=None, limit=None):
    data = manifest[:limit] if limit else manifest
    results = []; correct = 0; valid = 0; total_time = 0
    for i, entry in enumerate(data):
        img_path = entry['image']
        if not os.path.exists(img_path):
            print(f'  SKIP: {img_path}'); continue
        img_b64 = encode_image(img_path)
        gt = entry['class']
        t = time.time()
        try:
            if mode == 'few-shot' and ref_b64:
                raw = call_api(client, deployment, img_b64, make_prompt_fs_query(),
                               ref_b64=ref_b64, ref_prompt=make_prompt_fs_ref(), temperature=temperature)
            else:
                raw = call_api(client, deployment, img_b64, make_prompt_zs(), temperature=temperature)
        except Exception as e:
            raw = f'ERROR: {e}'
        elapsed = time.time() - t
        total_time += elapsed
        parsed = parse_response(raw)
        ok = False
        if parsed:
            valid += 1
            pred = parsed.get('defect_class', '').lower().strip()
            if pred == gt: ok = True; correct += 1
        results.append({'image': img_path, 'class': gt, 'predicted': parsed,
            'raw': raw, 'correct': ok, 'valid_json': parsed is not None,
            'time_s': round(elapsed, 2)})
        if (i+1) % 30 == 0:
            n = i+1
            print(f'  [{n}/{len(data)}] Acc: {correct}/{n} ({correct/n*100:.0f}%) | JSON: {valid}/{n}')
        time.sleep(0.25)
    return results, correct, valid, total_time

print('Benchmark function ready.')

Benchmark function ready.


## Quick Test (GPT-4.1 ZS)

In [11]:
quick = []
for cls in CLASSES:
    entry = next((e for e in manifest if e['class'] == cls), None)
    if entry: quick.append(entry)

print('Quick test — GPT-4.1 zero-shot')
print('=' * 55)
for entry in quick:
    img_b64 = encode_image(entry['image'])
    try:
        raw = call_api(client_41, GPT41_DEPLOYMENT, img_b64, make_prompt_zs(), temperature=0.7)
    except Exception as e:
        print(f'  {entry["class"]:>22} → ERROR: {e}'); continue
    parsed = parse_response(raw)
    gt = entry['class']
    pred = parsed.get('defect_class', '?') if parsed else '?'
    ok = '✓' if pred == gt else '✗'
    print(f'  {gt:>22} → {pred:<22} {ok}')
    time.sleep(0.3)

Quick test — GPT-4.1 zero-shot
     lack_of_penetration → lack_of_penetration    ✓
                porosity → porosity               ✓
                  cracks → no_defect              ✗
               no_defect → no_defect              ✓


---
## GPT-4.1 Full Benchmark

In [12]:
print('Running GPT-4.1 zero-shot...')
gpt41_zs_results, gpt41_zs_correct, gpt41_zs_valid, gpt41_zs_time = \
    run_frontier_benchmark(manifest, client_41, GPT41_DEPLOYMENT, 'zero-shot', temperature=0.7)
n = len(gpt41_zs_results)
print(f'\nGPT-4.1 ZS: Acc={gpt41_zs_correct}/{n} ({gpt41_zs_correct/n*100:.1f}%)')

Running GPT-4.1 zero-shot...
  [30/240] Acc: 18/30 (60%) | JSON: 30/30
  [60/240] Acc: 35/60 (58%) | JSON: 59/60
  [90/240] Acc: 56/90 (62%) | JSON: 89/90
  [120/240] Acc: 75/120 (62%) | JSON: 118/120
  [150/240] Acc: 91/150 (61%) | JSON: 147/150
  [180/240] Acc: 103/180 (57%) | JSON: 175/180
  [210/240] Acc: 121/210 (58%) | JSON: 203/210
  [240/240] Acc: 138/240 (57%) | JSON: 232/240

GPT-4.1 ZS: Acc=138/240 (57.5%)


In [13]:
print('Running GPT-4.1 few-shot...')
gpt41_fs_results, gpt41_fs_correct, gpt41_fs_valid, gpt41_fs_time = \
    run_frontier_benchmark(manifest, client_41, GPT41_DEPLOYMENT, 'few-shot', temperature=0.7, ref_b64=ref_b64)
n = len(gpt41_fs_results)
print(f'\nGPT-4.1 FS: Acc={gpt41_fs_correct}/{n} ({gpt41_fs_correct/n*100:.1f}%)')

Running GPT-4.1 few-shot...
  [30/240] Acc: 19/30 (63%) | JSON: 30/30
  [60/240] Acc: 39/60 (65%) | JSON: 60/60
  [90/240] Acc: 58/90 (64%) | JSON: 89/90
  [120/240] Acc: 81/120 (68%) | JSON: 119/120
  [150/240] Acc: 98/150 (65%) | JSON: 149/150
  [180/240] Acc: 114/180 (63%) | JSON: 179/180
  [210/240] Acc: 134/210 (64%) | JSON: 209/210
  [240/240] Acc: 156/240 (65%) | JSON: 239/240

GPT-4.1 FS: Acc=156/240 (65.0%)


---
## GPT-5 Full Benchmark

In [14]:
print('Running GPT-5 zero-shot...')
gpt5_zs_results, gpt5_zs_correct, gpt5_zs_valid, gpt5_zs_time = \
    run_frontier_benchmark(manifest, client_5, GPT5_DEPLOYMENT, 'zero-shot')
n = len(gpt5_zs_results)
print(f'\nGPT-5 ZS: Acc={gpt5_zs_correct}/{n} ({gpt5_zs_correct/n*100:.1f}%)')

Running GPT-5 zero-shot...
  [30/240] Acc: 20/30 (67%) | JSON: 30/30
  [60/240] Acc: 38/60 (63%) | JSON: 60/60
  [90/240] Acc: 59/90 (66%) | JSON: 90/90
  [120/240] Acc: 81/120 (68%) | JSON: 120/120
  [150/240] Acc: 96/150 (64%) | JSON: 150/150
  [180/240] Acc: 109/180 (61%) | JSON: 180/180
  [210/240] Acc: 128/210 (61%) | JSON: 210/210
  [240/240] Acc: 150/240 (62%) | JSON: 240/240

GPT-5 ZS: Acc=150/240 (62.5%)


In [15]:
print('Running GPT-5 few-shot...')
gpt5_fs_results, gpt5_fs_correct, gpt5_fs_valid, gpt5_fs_time = \
    run_frontier_benchmark(manifest, client_5, GPT5_DEPLOYMENT, 'few-shot', ref_b64=ref_b64)
n = len(gpt5_fs_results)
print(f'\nGPT-5 FS: Acc={gpt5_fs_correct}/{n} ({gpt5_fs_correct/n*100:.1f}%)')

Running GPT-5 few-shot...
  [30/240] Acc: 18/30 (60%) | JSON: 30/30
  [60/240] Acc: 36/60 (60%) | JSON: 60/60
  [90/240] Acc: 57/90 (63%) | JSON: 90/90
  [120/240] Acc: 81/120 (68%) | JSON: 120/120
  [150/240] Acc: 96/150 (64%) | JSON: 150/150
  [180/240] Acc: 109/180 (61%) | JSON: 180/180
  [210/240] Acc: 127/210 (60%) | JSON: 210/210
  [240/240] Acc: 150/240 (62%) | JSON: 240/240

GPT-5 FS: Acc=150/240 (62.5%)


---
## Results Summary

In [16]:
rows = []
for label, res, c, v, t in [
    ('GPT-4.1 (ZS, t=0.7)', gpt41_zs_results, gpt41_zs_correct, gpt41_zs_valid, gpt41_zs_time),
    ('GPT-4.1 (FS, t=0.7)', gpt41_fs_results, gpt41_fs_correct, gpt41_fs_valid, gpt41_fs_time),
    ('GPT-5 (ZS, t=1)',     gpt5_zs_results,  gpt5_zs_correct,  gpt5_zs_valid,  gpt5_zs_time),
    ('GPT-5 (FS, t=1)',     gpt5_fs_results,  gpt5_fs_correct,  gpt5_fs_valid,  gpt5_fs_time),
]:
    n = len(res)
    rows.append({'label': label, 'n': n, 'acc': round(c/n*100,1), 'json': round(v/n*100,1), 'time': round(t/n,2)})

# Load Qwen baselines if available
for mode in ['zero-shot', 'few-shot']:
    path = f'benchmark_results_{mode}.json'
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        rows.insert(0, {'label': f'Qwen2.5-VL-3B ({mode})', 'n': d['total_images'],
            'acc': d['accuracy_pct'], 'json': d['json_validity_pct'], 'time': d['avg_inference_time_s']})

print(f'{"Method":<28} {"N":>5} {"Accuracy":>10} {"JSON":>7} {"Time/img":>10}')
print('=' * 62)
for r in rows:
    print(f'{r["label"]:<28} {r["n"]:>5} {r["acc"]:>9.1f}% {r["json"]:>6.0f}% {r["time"]:>9.1f}s')
print(f'{"Random chance (4 classes)":<28} {"":>5} {100/4:>9.1f}%')

Method                           N   Accuracy    JSON   Time/img
Qwen2.5-VL-3B (few-shot)       240      51.2%    100%       2.9s
Qwen2.5-VL-3B (zero-shot)      240      30.8%    100%       2.0s
GPT-4.1 (ZS, t=0.7)            240      57.5%     97%       2.0s
GPT-4.1 (FS, t=0.7)            240      65.0%    100%       3.0s
GPT-5 (ZS, t=1)                240      62.5%    100%       5.6s
GPT-5 (FS, t=1)                240      62.5%    100%       8.0s
Random chance (4 classes)               25.0%


## Per-Class Breakdown

In [17]:
all_runs = [
    ('GPT-4.1 ZS', gpt41_zs_results),
    ('GPT-4.1 FS', gpt41_fs_results),
    ('GPT-5 ZS',   gpt5_zs_results),
    ('GPT-5 FS',   gpt5_fs_results),
]
for label, results in all_runs:
    print(f'\n{label}:')
    print(f'{"Class":<24} {"Correct":>10} {"Accuracy":>10}')
    print('-' * 46)
    by_class = defaultdict(list)
    for r in results: by_class[r['class']].append(r)
    for cls in CLASSES:
        cr = by_class[cls]
        c = sum(1 for r in cr if r['correct'])
        pct = c/len(cr)*100 if cr else 0
        print(f'{cls:<24} {c:>4}/{len(cr):<4} {pct:>9.1f}%')


GPT-4.1 ZS:
Class                       Correct   Accuracy
----------------------------------------------
lack_of_penetration        37/60        61.7%
porosity                   43/60        71.7%
cracks                      0/60         0.0%
no_defect                  58/60        96.7%

GPT-4.1 FS:
Class                       Correct   Accuracy
----------------------------------------------
lack_of_penetration        23/60        38.3%
porosity                   55/60        91.7%
cracks                     18/60        30.0%
no_defect                  60/60       100.0%

GPT-5 ZS:
Class                       Correct   Accuracy
----------------------------------------------
lack_of_penetration        43/60        71.7%
porosity                   47/60        78.3%
cracks                      0/60         0.0%
no_defect                  60/60       100.0%

GPT-5 FS:
Class                       Correct   Accuracy
----------------------------------------------
lack_of_penetration     

## Save Results

In [18]:
for label, results, correct, valid, tt, model_name in [
    ('gpt41_zs', gpt41_zs_results, gpt41_zs_correct, gpt41_zs_valid, gpt41_zs_time, 'gpt-4.1'),
    ('gpt41_fs', gpt41_fs_results, gpt41_fs_correct, gpt41_fs_valid, gpt41_fs_time, 'gpt-4.1'),
    ('gpt5_zs',  gpt5_zs_results,  gpt5_zs_correct,  gpt5_zs_valid,  gpt5_zs_time,  'gpt-5'),
    ('gpt5_fs',  gpt5_fs_results,  gpt5_fs_correct,  gpt5_fs_valid,  gpt5_fs_time,  'gpt-5'),
]:
    n = len(results)
    fname = f'benchmark_results_frontier_{label}.json'
    with open(fname, 'w') as f:
        json.dump({'model': model_name, 'mode': label, 'dataset': 'RIAWELC',
            'total_images': n, 'accuracy_pct': round(correct/n*100, 1),
            'json_validity_pct': round(valid/n*100, 1),
            'avg_inference_time_s': round(tt/n, 2),
            'results': results}, f, indent=2)
    print(f'Saved {fname}')

Saved benchmark_results_frontier_gpt41_zs.json
Saved benchmark_results_frontier_gpt41_fs.json
Saved benchmark_results_frontier_gpt5_zs.json
Saved benchmark_results_frontier_gpt5_fs.json
